# ridhorezkyanwar-testing.ipynb
# Testing & Prediction Request ke Wine Quality API

Notebook ini digunakan untuk menguji sistem machine learning yang sudah di-deploy di Railway.

Ganti `BASE_URL` dengan URL Railway Anda setelah deploy.

In [ ]:
import requests
import json
import pandas as pd

# Ganti dengan URL Railway Anda setelah deploy
BASE_URL = 'https://your-app.railway.app'

# Untuk testing lokal:
# BASE_URL = 'http://localhost:8080'

## Health Check

In [ ]:
response = requests.get(f'{BASE_URL}/health')
print(f'Status: {response.status_code}')
print(f'Response: {response.json()}')

## Single Prediction - Good Wine

In [ ]:
# wine berkualitas tinggi (quality >= 6)
good_wine = {
    'fixed_acidity': 7.4,
    'volatile_acidity': 0.28,
    'citric_acid': 0.34,
    'residual_sugar': 1.2,
    'chlorides': 0.045,
    'free_sulfur_dioxide': 35.0,
    'total_sulfur_dioxide': 141.0,
    'density': 0.9940,
    'pH': 3.42,
    'sulphates': 0.68,
    'alcohol': 12.5
}

response = requests.post(f'{BASE_URL}/predict', json=good_wine)
print(f'Status: {response.status_code}')
result = response.json()
print(f'Probability: {result["probability"]}')
print(f'Label: {result["label"]}')

## Single Prediction - Bad Wine

In [ ]:
# wine berkualitas rendah (quality < 6)
bad_wine = {
    'fixed_acidity': 11.2,
    'volatile_acidity': 0.92,
    'citric_acid': 0.01,
    'residual_sugar': 1.8,
    'chlorides': 0.075,
    'free_sulfur_dioxide': 17.0,
    'total_sulfur_dioxide': 60.0,
    'density': 0.9980,
    'pH': 3.16,
    'sulphates': 0.58,
    'alcohol': 9.8
}

response = requests.post(f'{BASE_URL}/predict', json=bad_wine)
print(f'Status: {response.status_code}')
result = response.json()
print(f'Probability: {result["probability"]}')
print(f'Label: {result["label"]}')

## Batch Prediction dari Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('data/wine_quality.csv')
sample = df.sample(10, random_state=42)

results = []
for _, row in sample.iterrows():
    payload = {
        'fixed_acidity': float(row['fixed_acidity']),
        'volatile_acidity': float(row['volatile_acidity']),
        'citric_acid': float(row['citric_acid']),
        'residual_sugar': float(row['residual_sugar']),
        'chlorides': float(row['chlorides']),
        'free_sulfur_dioxide': float(row['free_sulfur_dioxide']),
        'total_sulfur_dioxide': float(row['total_sulfur_dioxide']),
        'density': float(row['density']),
        'pH': float(row['pH']),
        'sulphates': float(row['sulphates']),
        'alcohol': float(row['alcohol']),
    }
    resp = requests.post(f'{BASE_URL}/predict', json=payload)
    pred = resp.json()
    actual = 'good' if row['quality'] >= 6 else 'bad'
    results.append({
        'actual_quality': int(row['quality']),
        'actual_label': actual,
        'predicted_label': pred.get('label'),
        'probability': pred.get('probability'),
        'correct': actual == pred.get('label')
    })

results_df = pd.DataFrame(results)
print(results_df.to_string())
print(f'\nAkurasi pada 10 sampel: {results_df["correct"].mean():.2%}')

## Cek Prometheus Metrics

In [ ]:
response = requests.get(f'{BASE_URL}/metrics')
print(f'Status: {response.status_code}')
# Tampilan baris yang relevan
for line in response.text.split('\n'):
    if 'prediction' in line and not line.startswith('#'):
        print(line)